# SCARLET Data Conversion Notebook

This notebook is intended for users preparing data reduction inputs.

Usage:
- set `input_dir`, `output_dir`, and `instrument_name` in the configuration cell;
- run the conversion cell to batch-convert raw files to SCARLET `NXsas_raw`;
- if an Excel workbook is available, run the last cell to display its first sheet in the notebook.


In [ ]:
from pathlib import Path
from html import escape

from IPython.display import HTML, display

from scarlet.io.converters import convert_to_scarlet_nxsas_raw, list_apparatus
from scarlet.reduction import reduce_2d
from scarlet.workflow.configuration import insert_masks_in_refs_file
from scarlet.io.xlsx import read_xlsx_rows


In [ ]:
input_dir = Path("data/SANSLLB/raw")
output_dir = Path("data/SANSLLB/processed")
instrument_name = "sansllb"

excel_path = output_dir / "run_configuration.xlsx"
entry_in = None
overwrite = False
validate_outputs = False
schema_name = "scarlet_nxsas_raw_v1.3_mono.yaml"

print("Known instruments:", ", ".join(list_apparatus()))


In [ ]:
RAW_SUFFIXES = (".nxs", ".h5", ".hdf", ".hdf5")
MULTI_SUFFIXES = (".nxs.h5",)


def base_name(path: Path) -> str:
    lower_name = path.name.lower()
    for suffix in (*MULTI_SUFFIXES, *RAW_SUFFIXES):
        if lower_name.endswith(suffix):
            return path.name[: -len(suffix)]
    return path.stem


def is_raw_candidate(path: Path) -> bool:
    lower_name = path.name.lower()
    if "_scarlet_nxsas_raw" in lower_name:
        return False
    return lower_name.endswith(MULTI_SUFFIXES) or path.suffix.lower() in RAW_SUFFIXES


def collect_input_files(folder: Path) -> list[Path]:
    return [path for path in sorted(folder.iterdir()) if path.is_file() and is_raw_candidate(path)]


def converted_output_path(input_path: Path, folder: Path) -> Path:
    return folder / f"{base_name(input_path)}_scarlet_nxsas_raw.nxs"


def display_table(rows: list[dict[str, object]]) -> None:
    if not rows:
        display(HTML("<p><em>No rows to display.</em></p>"))
        return

    headers = list(rows[0].keys())
    head_html = "".join(f"<th>{escape(str(header))}</th>" for header in headers)
    body_rows = []
    for row in rows:
        cells = "".join(f"<td>{escape(str(row.get(header, '')))}</td>" for header in headers)
        body_rows.append(f"<tr>{cells}</tr>")

    table_html = (
        "<table style='border-collapse: collapse; width: 100%;'>"
        "<thead><tr style='text-align: left; border-bottom: 2px solid #999;'>"
        f"{head_html}"
        "</tr></thead>"
        "<tbody>"
        f"{''.join(body_rows)}"
        "</tbody>"
        "</table>"
    )
    display(HTML(table_html))


In [ ]:
input_dir = input_dir.expanduser().resolve()
output_dir = output_dir.expanduser().resolve()
output_dir.mkdir(parents=True, exist_ok=True)

input_files = collect_input_files(input_dir)
if not input_files:
    raise FileNotFoundError(f"No raw input files found in {input_dir}")

schema = None
if validate_outputs:
    from scarlet.validation.schema_loader import load_schema
    from scarlet.validation.schema_validator import validate_nexus_file

    schema = load_schema(schema_name)

rows = []
for input_path in input_files:
    output_path = converted_output_path(input_path, output_dir)
    row = {
        "input_file": input_path.name,
        "output_file": output_path.name,
        "status": "ok",
        "entry_in": "auto" if entry_in is None else entry_in,
        "warnings": "",
        "validation": "",
    }
    try:
        report = convert_to_scarlet_nxsas_raw(
            instrument_name,
            input_path,
            output_path,
            entry_in=entry_in,
            overwrite=overwrite,
        )
        row["entry_in"] = report.entry_in
        row["warnings"] = "; ".join(report.warnings)
        if schema is not None:
            validation_report = validate_nexus_file(output_path, schema)
            row["validation"] = "OK" if validation_report.ok else "FAILED"
    except Exception as exc:
        row["status"] = "error"
        row["warnings"] = str(exc)
    rows.append(row)

display_table(rows)

ok_count = sum(row["status"] == "ok" for row in rows)
error_count = len(rows) - ok_count
print(f"Converted {ok_count} file(s) into {output_dir}")
if error_count:
    print(f"{error_count} conversion(s) failed. Inspect the table above.")


In [ ]:
excel_path = Path(excel_path).expanduser().resolve()
if excel_path.exists():
    print(f"Displaying workbook: {excel_path}")
    display_table(read_xlsx_rows(excel_path))
else:
    print(f"Excel workbook not found: {excel_path}")

mask_path = (input_dir / "mask_GQ.edf").resolve()
mask_detector_number = 0

if not mask_path.exists():
    raise FileNotFoundError(f"Mask file not found: {mask_path}")

refs_norm_files = sorted(output_dir.glob("refs_norm_*.nxs"))
if not refs_norm_files:
    raise FileNotFoundError(f"No refs_norm files found in {output_dir}")

mask_rows = []
for refs_norm_path in refs_norm_files:
    insert_masks_in_refs_file(
        refs_norm_path,
        detector_number=mask_detector_number,
        mask=mask_path,
    )
    mask_rows.append(
        {
            "refs_norm_file": refs_norm_path.name,
            "detector_number": mask_detector_number,
            "mask_file": mask_path.name,
            "status": "mask inserted",
        }
    )

display_table(mask_rows)


In [ ]:
sample_scattering_file = output_dir / "sans-llb2025n002403_scarlet_nxsas_raw.nxs"
sample_transmission_file = output_dir / "sans-llb2025n002402_scarlet_nxsas_raw.nxs"
refs_sub_file = output_dir / "refs_sub_config_1.nxs"
refs_norm_file = output_dir / "refs_norm_config_1.nxs"
reduced_output_file = output_dir / "reduced_2d" / "sample_reduced_2d.nxs"

reduce_detector_number = None
reduce_normalize_by = "monitor"
reduce_apply_mask = True
reduce_overwrite = True
reduce_azimuthal_bins = 200

reduced_output_file.parent.mkdir(parents=True, exist_ok=True)

result = reduce_2d(
    sample_scattering_file,
    refs_sub_file,
    sample_transmission=sample_transmission_file,
    refs_norm=refs_norm_file,
    output_path=reduced_output_file,
    detector_index=reduce_detector_number,
    normalize_by=reduce_normalize_by,
    apply_mask=reduce_apply_mask,
    overwrite=reduce_overwrite,
    azimuthal_bins=reduce_azimuthal_bins,
)

reduction_rows = [
    {
        "output_file": str(reduced_output_file.name),
        "sample_transmission": f"{result.sample_transmission.value:.6g}",
        "water_transmission": "" if result.water_transmission is None else f"{result.water_transmission.value:.6g}",
        "detectors": ", ".join(f"detector{i}" for i in result.detector_indices),
        "normalize_by": result.normalize_by,
    }
]
display_table(reduction_rows)
